#### **Welcome to Assignment 4 on Deep Learning for Computer Vision.**
This assignment is based on the content you learned in Week-4.

#### **Instructions**
1. Use Python 3.x to run this notebook
2. Write your code only in between the lines 'YOUR CODE STARTS HERE' and 'YOUR CODE ENDS HERE'. You should not change anything else in the code cells, if you do, the answers you are supposed to get at the end of this assignment might be wrong.
3. Read documentation of each function carefully.
4. All the Best!

We will be working with Pytorch operations

You will perform basic elementwise operations between two matrices

Step1: Rearrange the give input x into a tensor of shape (10,5,5) and store it in x_resize \\
Step2: Generate a gaussian random tensor (mean =0, variance=1) of shape (5,5) and store it in variable y1 \\
Step3:  Generate a uniform random tensor (interval [0,1]) of shape (5,5) and store it in variable y2 \\
Step4: Perform elementwise multiplication between x_resize and y1,such that the output will be of shape (10,5,5) and store it in variable mul_output1 \\
Step5: Sum over the last two dimensions of mul_output1 and store it in final_output1. The shape of final_output1 should be ([10]) \\
Step6: Repeat Step4 and Step5 for x_resize and y2

In [ ]:
import torch

# DO NOT CHANGE THE SEED
torch.manual_seed(2)

x: torch.Tensor = torch.randint(low=0, high=256, size=(1, 250))

## Your code here ##
# Step 1: Rearrange into shape (10, 5, 5)
x_resize: torch.Tensor = x.view(10, 5, 5)

# Step 2: Gaussian random tensor (mean=0, variance=1) of shape (5, 5)
y1: torch.Tensor = torch.randn(5, 5)

# Step 3: Uniform random tensor in [0, 1] of shape (5, 5)
y2: torch.Tensor = torch.rand(5, 5)

# Step 4: Elementwise multiplication with broadcasting -> shape (10, 5, 5)
mul_output1: torch.Tensor = x_resize * y1

# Step 5: Sum over last two dimensions -> shape (10,)
final_output1: torch.Tensor = torch.sum(mul_output1, dim=(-2, -1))

# Step 6: Repeat for x_resize and y2
mul_output2: torch.Tensor = x_resize * y2
final_output2: torch.Tensor = torch.sum(mul_output2, dim=(-2, -1))
## Your code ends here ##

print("Mean sum:", torch.mean(final_output1 + final_output2).item())


### Q1. What is mean of the sum of final_output1 and final_output2? (Choose the closest value)
a) 1000 \\
b) 2000 \\
c) 3000 \\
d) 4000

---

You will calculate L2 loss between predicted values and labels \\


Step1: Resize the final_output1 and final_output2 from the above code to shape (10,1) and store it in o1 and o2 respectively \\
Step2: Concatenate these two vectors o1 and o2 such that the shape of the resultant tensor is (10,2) and store it in final_output \\
Step3: Pass the tensor final_output as input to ReLU function and store it in logits variable \\
Step4: Calculate Eucledian norm of the difference (labels-logits)  across dimension = 1 and store it in eucledian_dist. Your output shape should be ([10])


In [ ]:
import torch

## DO NOT CHANGE SEED ##
torch.manual_seed(0)

labels: torch.Tensor = torch.randint(low=0, high=2000, size=(10, 2)).type(torch.float)

## Your code starts here ##
# Step 1: Resize final_output1 and final_output2 to (10, 1)
o1: torch.Tensor = final_output1.unsqueeze(1)
o2: torch.Tensor = final_output2.unsqueeze(1)

# Step 2: Concatenate into shape (10, 2)
final_output: torch.Tensor = torch.cat([o1, o2], dim=1)

# Step 3: Pass through ReLU activation
logits: torch.Tensor = torch.relu(final_output)

# Step 4: Calculate Euclidean norm across dimension 1
eucledian_dist: torch.Tensor = torch.norm(labels - logits, p=2, dim=1)
## Your code ends here ##

print("Euclidean distance shape:", eucledian_dist.shape)
loss: torch.Tensor = torch.sum(eucledian_dist)
print("Sum of Euclidean norm (loss):", loss.item())


### Q2. What is sum of the elements of the eucledian norm? (Choose the closest value)
a) 12000 \\
b) 13000 \\
c) 10000000 \\
d) 20000000

---

You will calculate gradient of a vector w.r.t to a function \\


Step1: Generate gaussian random tensor (mean =0, variance=1) of shape(3) and store it in w \\
Step2: Generate a uniform random tensor (interval [0,1]) of shape(3,3) and store it in X \\
Step3: Calculate the matrix vector product between X and w and store it in pred \\
Step4: Calculate L1 norm of the difference (labels - pred) and store it in temp. The output should be of shape [3] \\
Step5: Calculate sum of all elements of temp and store it in variable loss \\
Step6: Calculate the gradient of 'w' w.r.t function loss

In [ ]:
import torch

## DO NOT CHANGE SEED ##
torch.manual_seed(0)
labels: torch.Tensor = torch.randint(low=-800, high=50, size=(3, 1)).type(torch.float)

## Your code starts here ##
# Step 1: Gaussian random vector of shape (3,)
w: torch.Tensor = torch.randn(3)
## Your code ends here ##

w = w.requires_grad_()

## Your code starts here ##
# Step 2: Uniform random matrix in [0, 1] of shape (3, 3)
X: torch.Tensor = torch.rand(3, 3)

# Step 3: Matrix-vector product -> shape (3,)
pred: torch.Tensor = torch.matmul(X, w)

# Step 4: L1 norm of difference with broadcasting across dim=1 -> shape (3,)
temp: torch.Tensor = torch.norm(labels - pred, p=1, dim=1)

# Step 5: Sum over elements
loss: torch.Tensor = torch.sum(temp)
## Your code ends here ##

loss.backward(retain_graph=True)

## Your code starts here ##
# Step 6: Gradient of w w.r.t loss
grad = torch.autograd.grad(loss, w)
## Your code ends here ##
print("Gradient of w w.r.t loss:", grad)


### Q3. What is gradient of 'w' w.r.t loss function?
a)  [-4.4869, -2.1056, -3.4464] \\
b)  [4.4869, 2.1056, 3.4464] \\
c)  [-5.3670, -6.2075, -2.4481] \\
d)  [5.3670, 6.2075, 2.4481]

---

You will be training a 1-hidden layer multi-layer perceptron on a randomly generated dataset.

In [ ]:
# Please DO NOT modify this cell.

import os
import random
from typing import Optional
import numpy as np
import torch


def set_seed(seed: int) -> None:
    """Set random seeds across random, numpy, and PyTorch for deterministic execution.

    Args:
        seed: Random seed value.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


In [ ]:
# Please DO NOT modify this cell.

num_features = 10
classes = [0, 1, 2, 3, 4]
num_classes = len(classes)

num_samples = 100
num_train = 70
num_test = num_samples - num_train

In [ ]:
# Please DO NOT modify this cell.
# We are creating a random feature set and a random label set.
# The features and labels have no semantic meaning and might as well be garbage.

set_seed(2022)

features = np.random.random_sample((num_samples, num_features))
labels = np.random.choice(classes, size = num_samples)

# Train-test split
x_train = features[:num_train]
x_test = features[num_train:num_samples]

x_train = torch.Tensor(x_train)
x_test = torch.Tensor(x_test)

y_train = labels[:num_train]
y_test = labels[num_train:num_samples]

y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

In [ ]:
# Please DO NOT modify this cell.

print(f"Train features: {x_train.shape}")
print(f"Test features: {x_test.shape}")

print(f"Train labels: {y_train.shape}")
print(f"Train labels: {y_test.shape}")


Create train and test TensorDatasets from the respective numpy arrays

In [ ]:
from torch.utils.data import TensorDataset

#### YOUR CODE STARTS HERE ####
train_dataset: TensorDataset = TensorDataset(x_train, y_train)
test_dataset: TensorDataset = TensorDataset(x_test, y_test)
#### YOUR CODE ENDS HERE ####


Create dataloaders using the datasets created in the previous cell.
Use a batch size of 64

In [ ]:
from torch.utils.data import DataLoader

#### YOUR CODE STARTS HERE ####
batch_size: int = 64

train_loader: DataLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader: DataLoader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
#### YOUR CODE ENDS HERE ####


Network Definition

In [ ]:
# Please DO NOT modify this cell.

# Number of neurons in the hidden layer of our MLP
num_hidden = 512

Create the network MLP. Follow the comments to fill in the code

In [ ]:
import torch
import torch.nn as nn


class MLP(nn.Module):
    """Multi-Layer Perceptron classifier with a single hidden layer and ReLU activation.

    Attributes:
        hidden: Fully-connected projection from input features to hidden units.
        relu: Non-linear ReLU activation.
        classifier: Linear projection from hidden units to class logits.
    """

    def __init__(self, num_features: int, num_classes: int, num_hidden: int) -> None:
        """Initialize network layers.

        Args:
            num_features: Dimensionality of input feature vectors.
            num_classes: Number of distinct target classes.
            num_hidden: Number of hidden units in the intermediate layer.
        """
        super(MLP, self).__init__()

        #### YOUR CODE STARTS HERE ####
        self.hidden: nn.Linear = nn.Linear(num_features, num_hidden)
        self.relu: nn.ReLU = nn.ReLU()
        self.classifier: nn.Linear = nn.Linear(num_hidden, num_classes)
        #### YOUR CODE ENDS HERE ####

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the MLP.

        Args:
            x: Input tensor of shape (batch_size, num_features).

        Returns:
            Class logits of shape (batch_size, num_classes).
        """
        #### YOUR CODE STARTS HERE ####
        out: torch.Tensor = self.hidden(x)
        out = self.relu(out)
        out = self.classifier(out)
        #### YOUR CODE ENDS HERE ####
        return out


Training

In [ ]:
from torch.utils.data import DataLoader


def train(
    model: nn.Module,
    device: torch.device,
    train_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    epoch: int,
) -> float:
    """Train the model for a single epoch.

    Args:
        model: PyTorch module to optimize.
        device: Device on which execution takes place.
        train_loader: DataLoader for the training set.
        optimizer: Optimization algorithm.
        criterion: Loss function.
        epoch: Current epoch number.

    Returns:
        Average training loss over the dataset.
    """
    model.train()
    epoch_loss: float = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        #### YOUR CODE STARTS HERE ####
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output: torch.Tensor = model(data)
        loss: torch.Tensor = criterion(output, target)
        loss.backward()
        optimizer.step()
        #### YOUR CODE ENDS HERE ####

        epoch_loss += loss.item() * data.shape[0]

    avg_loss: float = epoch_loss / len(train_loader.dataset)
    if epoch % 50 == 0 or epoch == 1:
        print(f"Train Average Loss: {avg_loss:.2f}")
    return avg_loss


 Inference

In [ ]:
from typing import Tuple
from torch.utils.data import DataLoader


def test(
    model: nn.Module,
    device: torch.device,
    test_loader: DataLoader,
    criterion: nn.Module,
    mode: str = "Test",
) -> Tuple[float, float]:
    """Evaluate the model on test/train data.

    Args:
        model: PyTorch module to evaluate.
        device: Execution device.
        test_loader: DataLoader providing evaluation samples.
        criterion: Loss function.
        mode: Label for reporting ('Train' or 'Test').

    Returns:
        Tuple of (average_loss, accuracy_percentage).
    """
    model.eval()
    test_loss: float = 0.0
    correct: int = 0

    with torch.no_grad():
        for data, target in test_loader:
            #### YOUR CODE STARTS HERE ####
            data, target = data.to(device), target.to(device)
            output: torch.Tensor = model(data)
            #### YOUR CODE ENDS HERE ####

            test_loss += criterion(output, target).item() * data.shape[0]
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    test_acc: float = 100.0 * correct / len(test_loader.dataset)

    print(f"{mode} Average loss: {test_loss:.2f}")
    print(f"{mode} Accuracy: {correct}/{len(test_loader.dataset)} ({test_acc:.2f}%)")
    return test_loss, test_acc


In [ ]:
set_seed(2022)

num_epochs: int = 300

#### YOUR CODE STARTS HERE ####
device: torch.device = torch.device(
    "cuda:0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)

model: MLP = MLP(num_features, num_classes, num_hidden).to(device)
optimizer: torch.optim.Optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion: nn.Module = nn.CrossEntropyLoss()
#### YOUR CODE ENDS HERE ####

for epoch in range(1, num_epochs + 1):
    train(model, device, train_loader, optimizer, criterion, epoch)

print("--- Training complete ---")


In [ ]:
print(sum(p.numel() for p in model.parameters()))


### Q4. What are total number of parameters in the model?

a) 8197 \\
b) 18521 \\
c) 8356 \\
d) 9105

In [ ]:
test(model, device, train_loader, criterion, mode = "Train")

### Q5. Report the final train accuracy (Choose the closest option).

a) 58% \\
b) 93% \\
c) 100% \\
d) 89%

In [ ]:
test(model, device, test_loader, criterion, mode = "Test")

### Q6. Report the final test accuracy (Choose the closest option).

a) 30% \\
b) 40% \\
c) 10% \\
d) 70%